# Регрессия

# Автор: [Олег Булыгин](https://olegtalks.ru/)

<div style="background-color: #f5f5f5; border-left: 5px solid #3498db; padding: 10px; margin: 10px 0;">

## Обо мне

🐍 **Эксперт по Python и Data Science**. В 2020 году ушёл из найма, сейчас реализую собственные проекты, занимаюсь IT-образованием и консультированием.

👨‍💻 **Опыт в образовании**. В IT-образовании с 2017 года. За это время провел более 1500 лекций и вебинаров, обучил тысячи студентов, многие из которых сейчас работают в ведущих IT-компаниях. Создаю авторские курсы, консультирую крупные компании по вопросам Data Science и участвую в разработке образовательных программ для университетов и онлайн-школ.

🎯 Стремлюсь сделать обучение качественным и честным для всех, кто хочет развиваться в IT-сфере.

📱 Делюсь полезными материалами о Python:
- [Telegram](https://t.me/pythontalk)
- [Дзен](https://dzen.ru/pythontalk)
</div>


In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression, ElasticNet
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_validate, KFold, train_test_split
from sklearn.pipeline import make_pipeline
# !pip install phik
import phik
from phik.report import plot_correlation_matrix

In [ ]:
from sklearn.datasets import fetch_openml

data = fetch_openml(name='house_sales')
print(data.DESCR)

In [ ]:
df = data.data.drop(['date', 'zipcode', 'lat', 'long', 'yr_renovated'], axis=1)
df

In [ ]:
# !pip install ydata_profiling
from ydata_profiling import ProfileReport


profile = ProfileReport(df, title="Profiling Report")
profile

In [ ]:
corr_matrix = df.phik_matrix()
plot_correlation_matrix(corr_matrix.values, x_labels=corr_matrix.columns, y_labels=corr_matrix.index, figsize=(10, 8))

Видим, что между двумя признаками оказалась высокая корреляция.

Что тут можно сделать?
- Удалить один признак из такой пары:
- Попробовать применить **регуляризацию**.






 Если мы используем регуляризацию, то масштабирование признаков становится ещё более важным. Регуляризация накладывает штрафы на величины коэффициентов, и если признаки не отмасштабированы, модель может неправильно оценивать их важность.



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('price', axis=1),
                                                    df['price'],
                                                    random_state=42)

Для начала создадим обычную модель без полиномиальных признаков, чтобы сравнивать качество с ней.


## Пайплайны

Обратите внимание, что если мы осушествляем любой feature engineering, это должно происходить отдельно для каждой из итераций и каждый раз только на соответствующей обучающей выборке.
В нашем случае параметры масштабирования (среднее и стандартное отклонение) пассчитывались на всём `X_train`, а данные в каждом валидационном фолде уже масштабированы с использованием параметров, рассчитанных на `X_train`.
Это нарушает принцип кросс-валидации, так как валидационные данные (часть `X_train`, выделенная в каждом фолде) уже "знают" информацию о тренировочных данных, из которых были рассчитаны параметры масштабирования.

Таким, образом модель как бы "подглядывает" данные из трейна при оценке на валидации.

В sklearn для решения этой проблемы есть класс `Pipeline` и функция `make_pipeline`: в нём можно перечислить шаги работы с данными вплоть до применения модели.

In [ ]:
# Создаем пайплайн
pipeline = make_pipeline(StandardScaler(), LinearRegression())

# Обучаем пайплайн
pipeline.fit(X_train, y_train)

# Создаем KFold
kf = KFold(n_splits=3, shuffle=True, random_state=42)

# Выполняем кросс-валидацию
cv_metrics = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring='r2',
    return_train_score=True,
    n_jobs=-1
)


Посмотреть на результат всех преобразованйи можно так:

Берем срез `[:-1]`, чтобы исключить последний шаг пайплайна, который  является моделью (LinearRegression).

In [ ]:
X_train_transformed = pipeline[:-1].transform(X_train)
X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=X_train.columns)
X_train_transformed_df

И оцениваем качество:

In [ ]:
print(f"Среднее качество на тренировочной выборке: {cv_metrics['test_score'].mean()}")
print(f"Среднее качество на валидации: {cv_metrics['test_score'].mean()}")

При использовании пайпланой в каждой итерации кросс-валидации `StandardScaler` будет вычислять параметры масштабирования (среднее и стандартное отклонение) только на текущих тренировочных фолдах и применять их к текущим валидационным фолдам.

Это гарантирует, что тренировочные данные не будут "подглядывать" информацию из валидационных данных за пределами текущего фолда, что обеспечивает честную оценку модели.

## Полиномиальная регрессия



Полиномиальная регрессия (Polynomial Regression) — это более сложная модель, чем линейная регрессия. Вместо уравнения прямой используется уравнение полинома (многочлена), который отражает нелинейную зависимость. Степень полинома может быть сколь угодно большой: чем больше степень, тем сложнее модель.

В простом двумерном случае, когда мы рассматриваем зависимость целевого признака от одного фактора, полиномом второй степени будет уравнение параболы:

$$y = k_1 x + k_2 x^2 + b$$



Геометрически полином в двумерном пространстве — это некоторая кривая, которая пытается описать зависимость в данных. Выглядит это следующим образом:



Благодаря степенным слагаемым модель становится сложнее и начинает улавливать более сложные зависимости и выдавать меньшее смещение.

При реализации полиномиальной регрессии стоит проводить масштабирование **перед** генерацией полиномиальных признаков.

Это улучшает качество модели и обеспечивает более стабильные результаты при обучении.

Если делать масштабирование после создания полиномов, то:

- значения новых признаков могут значительно варьироваться. Стандартизация этих полиномов может сделать интерпретацию коэффициентов модели более сложной, так как значения полиномов уже изменены и не соответствуют исходным данным. Это затрудняет понимание того, как каждый признак влияет на зависимую переменную.

- модель станет более чувствительной к выбросам и шуму в данных. Поскольку полиномиальные признаки могут иметь большие значения, их стандартизация может не устранить проблемы переобучения, а наоборот, усугубить их, так как модель будет пытаться подстроиться под измененные масштабы.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

pipeline = make_pipeline(StandardScaler(),
                         PolynomialFeatures(degree=2, include_bias=False),
                         LinearRegression())

Посмотрим, как будут выглядеть тренировочные данные после преобразования в `pipeline`:

In [ ]:
pipeline.fit(X_train, y_train)

X_train_transformed = pipeline[:-1].transform(X_train)

# так мы получаем имена столбцов с полиномами
poly_feature_names = pipeline.named_steps['polynomialfeatures'].get_feature_names_out(input_features=X_train.columns)

X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=poly_feature_names)

X_train_transformed_df

In [ ]:
kf = KFold(3, shuffle=True, random_state=42)

cv_metrics = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring='r2',
    return_train_score=True,
    n_jobs=-1
)
cv_metrics

In [ ]:
print(f"Среднее качество на тренировочной выборке: {cv_metrics['train_score'].mean()}")
print(f"Среднее качество на валидационной выборке: {cv_metrics['test_score'].mean()}")

## Регуляризация

Методы регуляризации работают путем добавления штрафных коэффициентов к исходной функции потерь модели таким образом, что высокие значения коэффициентов снижаются. А признаки с очень низкими значениями коэффициентов (после штрафования) могут быть вообще отброшены. Это помогает уменьшать сложность модели, а также **снижает риск переобучения** путём уменьшения сложности модели.

**Штраф** — это дополнительное неотрицательное слагаемое в выражении для функции потерь, которое специально повышает ошибку.  За счёт этого слагаемого метод оптимизации будет находить не истинный минимум функции потерь, а псевдоминимум.

Существует множество методов регуляризации, из которых мы обсудим три наиболее часто используемых, а именно: Rigde, Lasso и Elastic Net.



Регуляризация Elastic Net сочетает в себе преимущества Lasso и Ridge-регуляризации в одной инструменте.

Функция потерь имеет два важных гиперпараметра: `alpha` и `l1_ratio`. Альфа в случае регуляризации Elastic Net — это константа, которая умножается на штрафы и для L1 (Lasso), и для L2 (Ridge). Гиперпараметр `l1_ratio` называется параметром смешивания таким образом, что `0 <= l1_ratio <= 1`.
Когда `l1_ratio` равен 1, это означает, что доля L1 (Lasso) равна 100%, а доля L2 (Ridge) равна 0%, то есть по факту просто делается Lasso-регуляризация. Аналогично, когда `l1_ratio` равно 0, это то же самое, что обычная Ridge-регуляризация.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

pipeline = make_pipeline(StandardScaler(),
                         PolynomialFeatures(degree=2, include_bias=False),
                         ElasticNet(random_state=42))

param_dist = {
    'elasticnet__alpha': np.linspace(0, 2000, 10),
    'elasticnet__l1_ratio': np.linspace(0, 1, 10)
}

random_search = RandomizedSearchCV(pipeline,
                                   param_distributions=param_dist,
                                   scoring='r2',
                                   n_iter=10,
                                   n_jobs=-1,
                                   cv=kf,
                                   random_state=42)

random_search.fit(X_train, y_train)

print(f"Наилучшее значение R2 при кросс-валидации: {random_search.best_score_}")
print(f"Наилучшие значения параметров: {random_search.best_params_}")

Оценим лучший результат на тесте:

In [ ]:
# Предсказываем значения
y_pred_enet = random_search.predict(X_test)

# Рассчитываем коэффициент детерминации
print(f"R^2 на тестовых данных: {r2_score(y_test, y_pred_enet)}")